# Read PDF and Build Presentation Summaries

This notebook is designed to read a PDF and extract the parts that are usually needed for research review work:

- full text
- page previews
- embedded images
- table-like data
- equation-like lines
- section-level summaries

It works best on digital PDFs. For scanned PDFs, run OCR first so equations and text are searchable.


## One-time setup

Install these packages once if they are not already available:

- `pip install pymupdf pandas matplotlib pillow`
- optional LLM summarization: `pip install openai`

Notes:

- `PyMuPDF` (`fitz`) is used for text, page rendering, embedded images, and tables when available.
- Page previews are saved for every page so figures and equations are still visible even when the PDF uses vector graphics instead of embedded raster images.


In [1]:
import json
import re
from collections import Counter
from pathlib import Path

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


In [ ]:
# Replace PDF_PATH with the file you want to analyze.
PDF_PATH = Path(r"C:\\path\\to\\PDF 1.pdf")
OUTPUT_DIR = Path(r"C:\\GIT\\r_proposal\\scripts\\pdf_outputs")

# Example configuration for the paper pasted in the prompt.
SECTIONS_TO_SUMMARIZE = ["3.1", "3.2", "3.3", "3.4"]
SECTION_PAGE_RANGE = list(range(21, 30))  # 1-based page numbers for the provided PDF 1 example

# Optional LLM summarization. Set USE_OPENAI = True only if the package is installed
# and OPENAI_API_KEY is available in your environment.
USE_OPENAI = False
OPENAI_MODEL = "gpt-4o-mini"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH, OUTPUT_DIR


In [ ]:
def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ").replace("\u2009", " ")
    text = re.sub(r"[ \\t]+", " ", text)
    return text.strip()


def extract_page_lines(page) -> list[str]:
    page_dict = page.get_text("dict")
    lines = []

    for block in page_dict.get("blocks", []):
        if block.get("type") != 0:
            continue

        for line in block.get("lines", []):
            spans = [span.get("text", "") for span in line.get("spans", [])]
            text = clean_text("".join(spans))
            if text:
                lines.append(text)

    return lines


def is_equation_like(line: str) -> bool:
    math_chars = "=+-*/^_∑∫√≤≥≈∞λμσθβπ"
    if any(ch in line for ch in math_chars):
        return True
    if re.search(r"\b(arg max|arg min|maximize|minimize)\b", line, flags=re.IGNORECASE):
        return True
    if re.search(r"\([0-9]+\)\s*$", line):
        return True
    if re.search(r"[A-Za-z]\s*=\s*[^=]", line):
        return True
    return False


def render_page_preview(page, page_number: int, preview_dir: Path, zoom: float = 2.0) -> Path:
    preview_dir.mkdir(parents=True, exist_ok=True)
    output_path = preview_dir / f"page_{page_number:03d}_preview.png"
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
    pix.save(output_path)
    return output_path


def extract_embedded_images(doc, page, page_number: int, image_dir: Path) -> list[dict]:
    image_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for idx, image_info in enumerate(page.get_images(full=True), start=1):
        xref = image_info[0]
        base_image = doc.extract_image(xref)
        ext = base_image.get("ext", "png")
        output_path = image_dir / f"page_{page_number:03d}_img_{idx:02d}.{ext}"
        output_path.write_bytes(base_image["image"])
        records.append({
            "page": page_number,
            "image_index": idx,
            "path": str(output_path),
            "width": base_image.get("width"),
            "height": base_image.get("height"),
            "ext": ext,
        })

    return records


def extract_tables_from_page(page, page_number: int, table_dir: Path) -> list[dict]:
    table_dir.mkdir(parents=True, exist_ok=True)
    records = []

    if not hasattr(page, "find_tables"):
        return records

    try:
        table_finder = page.find_tables()
    except Exception:
        return records

    for idx, table in enumerate(getattr(table_finder, "tables", []), start=1):
        data = table.extract()
        if not data:
            continue

        df = pd.DataFrame(data)
        output_path = table_dir / f"page_{page_number:03d}_table_{idx:02d}.csv"
        df.to_csv(output_path, index=False)

        records.append({
            "page": page_number,
            "table_index": idx,
            "path": str(output_path),
            "shape": df.shape,
        })

    return records


def extract_pdf_bundle(pdf_path: Path, output_dir: Path) -> dict:
    previews_dir = output_dir / "page_previews"
    images_dir = output_dir / "embedded_images"
    tables_dir = output_dir / "tables"

    pages = []
    image_records = []
    table_records = []

    with fitz.open(pdf_path) as doc:
        metadata = doc.metadata

        for page_index, page in enumerate(doc, start=1):
            lines = extract_page_lines(page)
            text = "\n".join(lines)
            equations = [line for line in lines if is_equation_like(line)]
            figures = [line for line in lines if re.match(r"^(Fig\.|Figure)\s*\d+", line)]
            table_captions = [line for line in lines if re.match(r"^Table\s*\d+", line)]

            preview_path = render_page_preview(page, page_index, previews_dir)
            page_images = extract_embedded_images(doc, page, page_index, images_dir)
            page_tables = extract_tables_from_page(page, page_index, tables_dir)

            image_records.extend(page_images)
            table_records.extend(page_tables)

            pages.append({
                "page": page_index,
                "text": text,
                "lines": lines,
                "equations": equations,
                "figure_captions": figures,
                "table_captions": table_captions,
                "preview_path": str(preview_path),
                "embedded_image_count": len(page_images),
                "table_count": len(page_tables),
            })

    full_text = "\n\n".join(page["text"] for page in pages)

    bundle = {
        "pdf_path": str(pdf_path),
        "output_dir": str(output_dir),
        "metadata": metadata,
        "pages": pages,
        "images": image_records,
        "tables": table_records,
        "full_text": full_text,
    }

    manifest_path = output_dir / "manifest.json"
    manifest_path.write_text(json.dumps(bundle, indent=2, ensure_ascii=False), encoding="utf-8")
    return bundle


def collect_assets_from_pages(bundle: dict, page_numbers: list[int]) -> dict:
    selected_pages = [page for page in bundle["pages"] if page["page"] in page_numbers]

    equations = []
    figure_captions = []
    table_captions = []
    preview_paths = []

    for page in selected_pages:
        equations.extend(page["equations"])
        figure_captions.extend(page["figure_captions"])
        table_captions.extend(page["table_captions"])
        preview_paths.append(page["preview_path"])

    page_tables = [table for table in bundle["tables"] if table["page"] in page_numbers]
    page_images = [image for image in bundle["images"] if image["page"] in page_numbers]

    return {
        "page_numbers": page_numbers,
        "equations": equations,
        "figure_captions": figure_captions,
        "table_captions": table_captions,
        "preview_paths": preview_paths,
        "tables": page_tables,
        "images": page_images,
    }


def show_page_previews(preview_paths: list[str], max_pages: int = 4, figsize=(14, 10)):
    preview_paths = preview_paths[:max_pages]
    if not preview_paths:
        print("No page previews to display.")
        return

    rows = len(preview_paths)
    fig, axes = plt.subplots(rows, 1, figsize=figsize)
    if rows == 1:
        axes = [axes]

    for ax, path in zip(axes, preview_paths):
        img = plt.imread(path)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(Path(path).name)

    plt.tight_layout()
    plt.show()


In [ ]:
STOPWORDS = {
    "the", "and", "of", "to", "a", "in", "for", "is", "that", "with", "as",
    "on", "this", "are", "be", "by", "an", "or", "from", "it", "at", "we",
    "can", "using", "used", "their", "which", "into", "also", "these", "our",
    "than", "was", "were", "have", "has", "had", "not", "but", "will", "they",
    "its", "more", "such", "may", "been", "one", "two", "three", "shown", "show",
}


def sentence_split(text: str) -> list[str]:
    text = clean_text(text).replace("\n", " ")
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9(])", text)
    return [s.strip() for s in sentences if len(s.strip()) > 25]


def extractive_summary(text: str, max_sentences: int = 6) -> str:
    sentences = sentence_split(text)
    if not sentences:
        return ""

    words = re.findall(r"[A-Za-z][A-Za-z0-9_-]+", text.lower())
    words = [word for word in words if word not in STOPWORDS and len(word) > 2]
    if not words:
        return " ".join(sentences[:max_sentences])

    freq = Counter(words)
    scores = []

    for idx, sentence in enumerate(sentences):
        sentence_words = re.findall(r"[A-Za-z][A-Za-z0-9_-]+", sentence.lower())
        score = sum(freq[word] for word in sentence_words if word in freq)
        scores.append((score, idx, sentence))

    top = sorted(scores, reverse=True)[:max_sentences]
    ordered = [sentence for _, _, sentence in sorted(top, key=lambda item: item[1])]
    return " ".join(ordered)


def next_same_level(section_number: str) -> str:
    parts = section_number.split(".")
    parts[-1] = str(int(parts[-1]) + 1)
    return ".".join(parts)


def next_major(section_number: str) -> str:
    return str(int(section_number.split(".")[0]) + 1)


def extract_section_by_number(full_text: str, section_number: str) -> str:
    peer = re.escape(next_same_level(section_number))
    major = re.escape(next_major(section_number))
    current = re.escape(section_number)

    pattern = re.compile(
        rf"(?ms)^\s*{current}\s+.*?(?=^\s*(?:{peer}|{major})\s+|\Z)"
    )
    match = pattern.search(full_text)
    return match.group(0).strip() if match else ""


def section_summaries(full_text: str, section_numbers: list[str]) -> dict:
    results = {}
    for section_number in section_numbers:
        section_text = extract_section_by_number(full_text, section_number)
        results[section_number] = {
            "text": section_text,
            "extractive_summary": extractive_summary(section_text, max_sentences=6),
        }
    return results


def summarize_with_openai(section_text: str, model: str = OPENAI_MODEL) -> str | None:
    if not USE_OPENAI:
        return None

    try:
        from openai import OpenAI
    except ImportError:
        print("Install the openai package to use LLM summarization.")
        return None

    client = OpenAI()
    prompt = f"""
Create a detailed presentation-ready summary of the following paper section.
Focus on: motivation, method, equations, dataset, figures, results, limitations, and speaker notes.

Section text:
{section_text}
"""

    response = client.responses.create(
        model=model,
        input=prompt,
    )
    return response.output_text


In [ ]:
bundle = None

if PDF_PATH.exists():
    bundle = extract_pdf_bundle(PDF_PATH, OUTPUT_DIR)
    page_index = pd.DataFrame([
        {
            "page": page["page"],
            "chars": len(page["text"]),
            "equation_lines": len(page["equations"]),
            "figure_captions": len(page["figure_captions"]),
            "embedded_images": page["embedded_image_count"],
            "tables": page["table_count"],
        }
        for page in bundle["pages"]
    ])

    display(Markdown(f"**Loaded:** `{PDF_PATH}`"))
    display(Markdown(f"**Output folder:** `{OUTPUT_DIR}`"))
    display(page_index.head(10))
else:
    print(f"Set PDF_PATH to an existing PDF file. Current value not found: {PDF_PATH}")


In [ ]:
if bundle is not None:
    asset_slice = collect_assets_from_pages(bundle, SECTION_PAGE_RANGE)
    section_map = section_summaries(bundle["full_text"], SECTIONS_TO_SUMMARIZE)

    display(Markdown("## Section summaries"))
    for section_number, section_data in section_map.items():
        display(Markdown(f"### Section {section_number}"))
        display(Markdown(section_data["extractive_summary"] or "No text found for this section."))

    display(Markdown("## Equation-like lines found in the selected pages"))
    for line in asset_slice["equations"][:20]:
        print(line)

    display(Markdown("## Figure captions found in the selected pages"))
    for line in asset_slice["figure_captions"]:
        print(line)

    display(Markdown("## Saved table files"))
    if asset_slice["tables"]:
        display(pd.DataFrame(asset_slice["tables"]))
    else:
        print("No tables detected in the selected pages.")

    show_page_previews(asset_slice["preview_paths"], max_pages=3)


## Example presentation summary for PDF 1: Sections 3.1 to 3.4

### Slide 1: Why this case study matters
- The paper uses lithium-ion battery retirement as a concrete example of what a complete digital twin should do.
- The motivation is practical: battery cells degrade differently depending on how they were used in first life, so a fixed fleet-wide retirement rule wastes value.
- The authors argue that many so-called battery digital twins stop at prognosis. They estimate remaining useful life, but they do not close the loop with optimization and action.
- Their key contribution is a five-dimensional digital twin that not only predicts degradation, but also decides when to retire a cell from first life for second-life use.

### Slide 2: What makes this a true digital twin
- The case study is explicitly framed around all five dimensions from Part 1: physical system, physical-to-virtual data flow, digital system, virtual-to-physical prediction, and optimization.
- Software component 1 is a particle-filter prognostic model. It updates the battery state online from incoming capacity data and predicts future degradation and RUL.
- Software component 2 is a multi-attribute utility optimizer. It uses the prognostic forecast to recommend the best retirement cycle.
- The loop is completed when the optimization output triggers an operational decision by personnel.

### Slide 3: Section 3.1 Background
- Li-ion cells lose capacity because of irreversible electrochemical changes. This makes future capacity fade and RUL prediction central to battery management.
- Interest in second-life batteries is increasing, but second-life value depends strongly on current health, which itself depends on each cell's first-life history.
- Because of this history dependence, the authors reject simple retirement rules such as removing every EV battery after the same mileage threshold.
- Their main thesis is that retirement must be optimized online at the cell or pack level.
- This section also makes an important conceptual point: a prognostic model alone is not a full digital twin unless it feeds an optimization or control action.

### Slide 4: Section 3.2 Methods overview
- Figure 9 shows the workflow. Offline run-to-failure battery data is used first to tune the particle filter.
- During online operation, new battery capacity measurements update the particle filter recursively.
- When measured capacity drops to 95% of the initial value, the retirement optimizer is triggered.
- The optimizer evaluates future utility using the predicted capacity trajectory and selects the best retirement cycle in advance.
- Figure 10 shows the offline/online split and the train-test structure used in the experiments.

### Slide 5: Section 3.2.1 Prognostic model
- The authors choose an empirical power-law degradation model because battery fade curves can take different shapes and a compact model is needed for online filtering.
- The measurement model is a power-law relation between normalized capacity and cycle number: `Q_k = 1 - a k^b`.
- The unknown parameters `a` and `b` evolve through random-walk state equations: `a_{k+1} = a_k + w_a` and `b_{k+1} = b_k + w_b`.
- A particle filter is used instead of a Kalman-filter variant because the authors want flexibility, fast convergence, and non-parametric RUL distributions rather than a purely Gaussian assumption.
- This matters for a digital twin because it produces both state estimates and uncertainty-aware forecasts that can be passed to later decision layers.

### Slide 6: Section 3.2.2 Optimization model
- The optimization layer is based on multi-attribute utility theory. The key idea is to convert different decision criteria onto the same utility scale and then optimize a combined score.
- The first attribute is total discharge ampere-hours before retirement. This rewards using the cell as much as possible in first life.
- The second attribute is mean time between charges. This represents customer experience because a healthier battery gives longer driving range between charges.
- These two objectives conflict: delaying retirement increases first-life throughput, but it also degrades range and customer utility.
- The final utility is an equal-weight combination: `Lambda = 0.5 * phi_1(Ah(x_c)) + 0.5 * phi_2(MTBC(x_c))`, where `x_c` is the candidate retirement cycle.

### Slide 7: Section 3.3.1 Dataset
- The experiments use the open-source MIT battery dataset from Severson et al.
- The dataset contains 124 lithium iron phosphate/graphite cells tested under different two-step fast-charging policies.
- All cells are discharged at 4C, and the charging-policy variation creates large diversity in lifetime and degradation shape.
- The authors split the data into 41 training cells, 43 primary test cells, and 40 secondary test cells.
- To model a realistic second-life decision problem, they extend each trajectory beyond first life and linearly extrapolate the last 30 measurements until 50% normalized capacity.

### Slide 8: Section 3.3.2 Prognostic results
- The particle filter hyperparameters are set from the training set: measurement noise at 0.01, both process noises at 0.05, and the initial `a` and `b` values at the dataset medians.
- Figure 12 shows that the particle filter can generate a full future capacity trajectory and an empirical end-of-life distribution, not just a point prediction.
- Figure 14 shows a systematic issue: the model tends to underestimate RUL, especially for very long-life cells.
- The paper attributes this to three main causes: the late-life linear extrapolation conflicts with the power-law model, the model is initialized around median-life cells, and the test-set lifetime distribution is longer than the training-set distribution.
- In short, the prognostic layer works, but generalization degrades under distribution shift and model mismatch.

### Slide 9: Section 3.3.3 Optimization results
- The two utility functions are exponential and mapped to the range `[0, 1]`.
- For total ampere-hour throughput, the utility parameters are chosen around the observed training-set range, with lower and upper bounds of 300 Ah and 1000 Ah and a risk parameter of 200.
- For mean time between charges, the bounds are 0.21 h and 0.25 h with a risk parameter of 0.015.
- Figures 17 to 19 show that the optimizer behaves differently across cells. For cells with lifetimes close to the training median, it finds a reasonable balance between throughput and user experience.
- For very long-life cells, the total-Ah utility saturates early, so the decision is dominated by the charging-interval utility, which pushes retirement too close to the current cycle.

### Slide 10: Main interpretation of the optimization behavior
- The optimal retirement cycle is closely tied to the knee point of the degradation curve, which is where capacity starts to fall faster.
- That makes intuitive sense: before the knee point, keeping the battery in service still yields good throughput without too much user penalty.
- After the knee point, each extra cycle gives less first-life value and more customer inconvenience because charging becomes more frequent.
- The optimizer is therefore really trading off two notions of value: extracting more work now versus preserving useful quality during service.
- The quality of the retirement decision depends more on the overall degradation trend and slope than on getting every future point exactly right.

### Slide 11: Section 3.4 Conclusion and future work
- The case study shows how a maintenance-oriented battery digital twin can be built by integrating forecasting and optimization.
- The authors see the particle filter as a solid first step, but they note that a single power-law model is too restrictive. A multi-model particle filter is proposed as future work.
- They also acknowledge that the utility design is fragile for cells with much longer lifetimes than the training set. More general utility design is needed.
- The current optimization is deterministic even though the particle filter produces probabilistic outputs. A stronger next step would be probabilistic retirement intervals rather than single retirement points.
- They also want future versions to incorporate second-life stakeholder preferences, not just first-life EV usage objectives.

### Slide 12: Key takeaways for your presentation
- The case study is valuable because it moves from prediction to decision-making, which is the real differentiator of a digital twin.
- Its architecture is simple enough to understand and reproduce, yet rich enough to show how online data assimilation and optimization can be linked.
- The strongest technical idea is the coupling of a particle-filter degradation forecast with a utility-based retirement decision rule.
- The main weakness is that both the degradation model and the utility design are sensitive to distribution shift and to assumptions baked in during offline tuning.
- For a presentation, the core message is: the paper demonstrates that a battery digital twin becomes most useful when uncertainty-aware forecasting is connected directly to operational decisions.

### Suggested speaker close
- This battery example is not just about batteries. It is a template for digital twins in other maintenance settings: estimate the evolving state, forecast future degradation, quantify uncertainty, and optimize the action before failure or value loss occurs.
